In [1]:
import os
import sys

if "google.colab" in sys.modules:
    # Running in Colab

    !git clone https://github.com/pthengtr/kcw-analytics.git
    !cd /content/kcw-analytics && git pull origin main

    from google.colab import drive
    drive.mount("/content/drive")

    BASE_FOLDER = "/content/drive/Shareddrives"
    BASE_FOLDER_GIT = "/content"
else:
    # Running in local Jupyter
    BASE_FOLDER = r"G:\Shared drives"
    BASE_FOLDER_GIT = r"C:\Users\Windows 11\Notebook"

print("Using folder:", BASE_FOLDER)

Cloning into 'kcw-analytics'...
remote: Enumerating objects: 706, done.
remote: Counting objects: 100% (65/65), done.
remote: Compressing objects: 100% (52/52), done.
remote: Total 706 (delta 42), reused 24 (delta 13), pack-reused 641 (from 2)
Receiving objects: 100% (706/706), 595.19 KiB | 6.54 MiB/s, done.
Resolving deltas: 100% (475/475), done.
From https://github.com/pthengtr/kcw-analytics
 * branch            main       -> FETCH_HEAD
Already up to date.
Mounted at /content/drive
Using folder: /content/drive/Shareddrives


In [2]:
folder = f"{BASE_FOLDER}/KCW-Data/kcw_analytics/01_raw"

In [4]:
import os
import pandas as pd

data = {}

for file in os.listdir(folder):
    if file.endswith(".csv"):
        path = os.path.join(folder, file)
        data[file] = pd.read_csv(
            path,
            dtype={
              "BCODE": "string",
              "ITEMNO": "string",
              "BILLNO": "string",
            },
            encoding="utf-8-sig",
            low_memory=False   # stops chunk guessing
        )
        print(f"Loaded: {file} -> {data[file].shape}")

Loaded: raw_inventory_hq_2024.csv -> (4983, 8)
Loaded: raw_syp_pimas_purchase_bills.csv -> (2983, 49)
Loaded: raw_syp_pidet_purchase_lines.csv -> (27880, 41)
Loaded: raw_syp_sidet_sales_lines.csv -> (38127, 38)
Loaded: raw_syp_simas_sales_bills.csv -> (13021, 49)
Loaded: raw_hq_pidet_purchase_lines.csv -> (154087, 41)
Loaded: raw_hq_pimas_purchase_bills.csv -> (50269, 49)
Loaded: raw_hq_icmas_products.csv -> (114973, 94)
Loaded: raw_hq_sidet_sales_lines.csv -> (733591, 38)
Loaded: raw_hq_simas_sales_bills.csv -> (276119, 49)
Loaded: raw_hq_apmas_payable.csv -> (978, 20)
Loaded: raw_hq_armas_receivable.csv -> (2228, 20)
Loaded: raw_hq_pvmas_notes_vouchers.csv -> (13875, 32)
Loaded: raw_hq_rvmas_notes_vouchers.csv -> (13875, 32)


In [6]:
folder = f"{BASE_FOLDER}/KCW-Data/kcw_analytics/01_raw/statement"

In [7]:
!pip install xlrd

In [8]:
import os
import pandas as pd

data_statement = {}

for file in os.listdir(folder):
    path = os.path.join(folder, file)
    ext = os.path.splitext(file)[1].lower()

    try:
        if ext == ".csv":
            # text file
            last_err = None
            for enc in ["utf-8-sig", "cp874", "cp1252", "latin1"]:
                try:
                    df = pd.read_csv(path, skiprows=10, encoding=enc, low_memory=False)
                    source = f"csv-{enc}"
                    break
                except Exception as e:
                    last_err = e
            else:
                raise last_err

        elif ext == ".xlsx":
            # modern Excel
            df = pd.read_excel(path, header=10, engine="openpyxl")
            source = "xlsx-openpyxl"

        elif ext == ".xls":
            # old Excel binary format
            df = pd.read_excel(path, header=10, engine="xlrd")
            source = "xls-xlrd"

        else:
            continue

        for col in ["BCODE", "ITEMNO", "BILLNO"]:
            if col in df.columns:
                df[col] = df[col].astype("string")

        key = os.path.splitext(file)[0]
        data_statement[key] = df
        print(f"Loaded: {file} -> {df.shape} ({source})")

    except Exception as e:
        print(f"Failed: {file} -> {type(e).__name__}: {e}")

Loaded: raw_statement_6184.xls -> (31, 9) (xls-xlrd)
Loaded: raw_statement_3557.csv -> (138, 13) (csv-utf-8-sig)
Loaded: raw_statement_7236.csv -> (92, 13) (csv-utf-8-sig)
Loaded: raw_statement_1139.xls -> (42, 9) (xls-xlrd)
Loaded: raw_statement_0393.csv -> (67, 13) (csv-utf-8-sig)


In [9]:
import sys
import importlib

# ensure repo is on path
repo_path = f"{BASE_FOLDER_GIT}/kcw-analytics"
if repo_path not in sys.path:
    sys.path.append(repo_path)

# import the module (NOT individual functions)
import src.kcw.utils as utils

# reload to pick up latest .py changes
importlib.reload(utils)


<module 'src.kcw.utils' from '/content/kcw-analytics/src/kcw/utils.py'>

In [35]:
df_3557 = data_statement["raw_statement_3557"].copy()

In [36]:
df_pimas = data["raw_hq_pimas_purchase_bills.csv"].copy()
df_pvmas = data["raw_hq_pvmas_notes_vouchers.csv"].copy()

In [39]:
import pandas as pd

def filter_last_1year(df, date_col="BILLDATE"):
    df = df.copy()
    df[date_col] = pd.to_datetime(df[date_col], errors="coerce")

    cutoff = pd.Timestamp.today() - pd.DateOffset(years=1)

    return df[df[date_col] >= cutoff]

df_pimas = filter_last_1year(df_pimas)
df_pvmas = filter_last_1year(df_pvmas, date_col="VOUCDATE")

In [53]:
import pandas as pd

def match_billno_candidates_by_amount(
    df_source: pd.DataFrame,
    df_statement: pd.DataFrame,
    *,
    source_name: str,
    source_amount_col: str = "AMOUNT",
    statement_amount_col: str = "Amount",
    source_billno_col: str = "BILLNO",
    source_acctname_col: str | None = "ACCTNAME",
    tolerance: float = 1.0,
    max_matches: int = 5,
    out_col: str | None = None,
    sort_by_amount_diff: bool = True,
    copy: bool = True,
    verbose: bool = True,
) -> pd.DataFrame:
    """
    For each statement row, find up to max_matches source rows within +/- tolerance
    on amount, then output all matches into one text cell.

    Example cell output:
        BILLNO1 : ACCTNAME1
        BILLNO2 : ACCTNAME2
        BILLNO3 : ACCTNAME3
    """

    source = df_source.copy()
    stmt = df_statement.copy() if copy else df_statement

    source[source_amount_col] = pd.to_numeric(
        source[source_amount_col].astype(str).str.replace(",", "", regex=False),
        errors="coerce"
    )
    stmt[statement_amount_col] = pd.to_numeric(
        stmt[statement_amount_col].astype(str).str.replace(",", "", regex=False),
        errors="coerce"
    )

    stmt["_row_id"] = range(len(stmt))

    source_valid = source.dropna(subset=[source_amount_col]).copy()
    stmt_valid = stmt.dropna(subset=[statement_amount_col]).copy()

    if out_col is None:
        out_col = f"{source_name}.MATCH_CANDIDATES"

    def build_candidate_text(stmt_amount):
        candidates = source_valid[
            source_valid[source_amount_col].sub(stmt_amount).abs() <= tolerance
        ].copy()

        if candidates.empty:
            return pd.NA

        candidates["_amount_diff"] = (candidates[source_amount_col] - stmt_amount).abs()

        if sort_by_amount_diff:
            sort_cols = ["_amount_diff"]
            ascending = [True]
            if source_billno_col in candidates.columns:
                sort_cols.append(source_billno_col)
                ascending.append(True)
            candidates = candidates.sort_values(sort_cols, ascending=ascending)

        candidates = candidates.head(max_matches)

        lines = []
        for _, row in candidates.iterrows():
            billno = row.get(source_billno_col, "")
            acctname = row.get(source_acctname_col, "") if source_acctname_col else ""

            billno = "" if pd.isna(billno) else str(billno)
            acctname = "" if pd.isna(acctname) else str(acctname)

            if source_acctname_col:
                lines.append(f"{billno} : {acctname}")
            else:
                lines.append(billno)

        return "\n".join(lines) if lines else pd.NA

    matched = stmt_valid[["_row_id", statement_amount_col]].copy()
    matched[out_col] = matched[statement_amount_col].apply(build_candidate_text)

    out = stmt.merge(
        matched[["_row_id", out_col]],
        on="_row_id",
        how="left",
    )

    out = out.sort_values("_row_id").drop(columns=["_row_id"])

    if verbose:
        print(
            f"[match_billno_candidates_by_amount] "
            f"matched_rows={out[out_col].notna().sum():,}/{len(out):,}"
        )

    return out

import pandas as pd

def match_multiple_sources_candidates_by_amount(
    df_statement: pd.DataFrame,
    source_configs: list[dict],
    *,
    statement_amount_col: str = "Amount",
    tolerance: float = 1.0,
    max_matches: int = 5,
) -> pd.DataFrame:
    """
    Apply candidate-list amount matching repeatedly using a list of source configs.

    Each config can look like:
    {
        "df_source": df_pvmas,
        "source_name": "vouchers",
        "source_amount_col": "BILLAMT",
        "source_billno_col": "VOUCNO",
        "source_acctname_col": "ACCTNAME",
        "statement_amount_col": "ถอนเงิน",     # optional
        "tolerance": 1.0,                      # optional
        "max_matches": 5,                      # optional
        "out_col": "voucher_candidates",       # optional
    }
    """
    out = df_statement.copy()

    for cfg in source_configs:
        out = match_billno_candidates_by_amount(
            df_source=cfg["df_source"],
            df_statement=out,
            source_name=cfg["source_name"],
            source_amount_col=cfg.get("source_amount_col", "AMOUNT"),
            statement_amount_col=cfg.get("statement_amount_col", statement_amount_col),
            source_billno_col=cfg.get("source_billno_col", "BILLNO"),
            source_acctname_col=cfg.get("source_acctname_col", "ACCTNAME"),
            tolerance=cfg.get("tolerance", tolerance),
            max_matches=cfg.get("max_matches", max_matches),
            out_col=cfg.get("out_col"),
            sort_by_amount_diff=cfg.get("sort_by_amount_diff", True),
            copy=False,
            verbose=cfg.get("verbose", True),
        )

    return out

In [51]:
import pandas as pd

def build_statement_datetime(
    df: pd.DataFrame,
    *,
    date_col: str = "วันที่",
    time_col: str = "เวลา/ วันที่มีผล",
    out_col: str = "date",
    drop_original: bool = False,
    copy: bool = True,
    verbose: bool = True,
) -> pd.DataFrame:
    """
    Merge Thai bank statement date + time columns into a single datetime column.

    Example input:
        วันที่           เวลา/ วันที่มีผล
        02-02-26        08:32
        02-02-26        NaN

    Output:
        date
        2026-02-02 08:32
        2026-02-02 00:00
    """

    df = df.copy() if copy else df

    # clean column names
    df.columns = df.columns.str.strip()

    # remove Excel unnamed columns
    df = df.loc[:, ~df.columns.str.contains("^Unnamed")]

    if date_col not in df.columns:
        raise ValueError(f"{date_col} not found in dataframe")

    if time_col not in df.columns:
        raise ValueError(f"{time_col} not found in dataframe")

    df[out_col] = pd.to_datetime(
        df[date_col].astype(str) + " " + df[time_col].fillna("00:00").astype(str),
        format="%d-%m-%y %H:%M",
        errors="coerce"
    )

    if drop_original:
        df.drop(columns=[date_col, time_col], inplace=True)

    if verbose:
        print(
            f"[build_statement_datetime] created '{out_col}' "
            f"valid={df[out_col].notna().sum():,}/{len(df):,}"
        )

    return df

In [42]:
# Clean unamed column
df_3557 = df_3557.loc[:, ~df_3557.columns.str.contains("^Unnamed")]

In [43]:
df_3557 = build_statement_datetime(df_3557, drop_original=True)

[build_statement_datetime] created 'date' valid=138/138


In [54]:
source_configs = [
    {
        "df_source": df_pimas,
        "source_name": "purchases",
        "source_amount_col": "AFTERTAX",
        "statement_amount_col": "ถอนเงิน",
        "extra_source_cols": ["ACCTNO", "ACCTNAME"],
    },
    {
        "df_source": df_pvmas,
        "source_name": "vouchers",
        "source_amount_col": "NETAMT",
        "source_billno_col": "VOUCNO",
        "statement_amount_col": "ถอนเงิน",
        "extra_source_cols": ["ACCTNO", "ACCTNAME"],
    },
]

df_out_3557 = match_multiple_sources_candidates_by_amount(
    df_statement=df_3557,
    source_configs=source_configs,
    tolerance=1.0,
)

[match_billno_candidates_by_amount] matched_rows=71/138
[match_billno_candidates_by_amount] matched_rows=85/138


In [55]:
df_out_3557.columns

Index(['รายการ', 'ถอนเงิน', 'ฝากเงิน', 'ยอดคงเหลือ', 'ช่องทาง', 'รายละเอียด',
       'date', 'purchases.MATCH_CANDIDATES', 'vouchers.MATCH_CANDIDATES'],
      dtype='object')

In [56]:
df_out_3557[['date', 'รายการ', 'ถอนเงิน',
       'ฝากเงิน','ยอดคงเหลือ', 'รายละเอียด', 'purchases.MATCH_CANDIDATES', 'vouchers.MATCH_CANDIDATES']]

,date,รายการ,ถอนเงิน,ฝากเงิน,ยอดคงเหลือ,รายละเอียด,purchases.MATCH_CANDIDATES,vouchers.MATCH_CANDIDATES
0,2026-02-01 00:00:00,ยอดยกมา,NaN,NaN,"266,827.66",NaN,NaN,NaN
1,2026-02-02 08:32:00,โอนเงิน,3477.5,NaN,"263,350.16",โอนไป BBL X2440 บจ.เค.ที. อินเตอร์++,-202512/01750 : บริษัท เดชณรงค์ แทรคเตอร์ จำกั...,<NA>
2,2026-02-02 09:37:00,รับโอนเงิน,NaN,"90,000.00","353,350.16",จาก X7236 บจก. เกียรติชัยอะไ++,NaN,NaN
3,2026-02-02 11:33:00,โอนเงิน,6850.0,NaN,"346,500.16",โอนไป X7775 หจก. ตะวันออกศูนย์++,030122 : ห้างหุ้นส่วนจำกัด บี เอ็ม แทรกเตอร์\n...,<NA>
4,2026-02-02 13:35:00,โอนเงิน,17000.0,NaN,"329,500.16",โอนไป X8485 นาย วิศาล อรรจนาว++,633130 : ร้าน ชัยมอเตอร์ พุเตย\nIV68060316 : บ...,<NA>
...,...,...,...,...,...,...,...,...
133,2026-02-28 15:50:00,โอนเงิน,26980.0,NaN,"339,952.51",โอนไป X1513 บจก. เอส.เอ.อี.ออโ++,<NA>,P6902-079 : บริษัท เอส.เอ.อี ออโต้แทรค จำกัด (...
134,2026-02-28 15:51:00,โอนเงิน,14990.0,NaN,"324,962.51",โอนไป SCB X3290 ห้างหุ้นส่วนจำกัด ++,<NA>,P6902-078 : ห้างหุ้นส่วนจำกัด พี.อี.ออโต้เทรด ...
135,2026-02-28 15:58:00,โอนเงิน,208280.0,NaN,"116,682.51",โอนไป X8989 บจก. สามมิตร โอโตพ++,<NA>,P6902-081 : บริษัท สามมิตรโอโตพาร์ท จำกัด (สำน...
136,2026-02-28 16:01:00,รับโอนเงิน,NaN,"100,000.00","216,682.51",จาก X0393 บจก. เกียรติชัยอะไ++,NaN,NaN
